Laod data sets

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd

#Processed movies for content-based
movies_df = pd.read_csv(r"C:\Users\Dell\Documents\My ML Projects\Hybrid Recommendation System\data\processed_movies.csv")

#Ratings for collaborative filtering
df = pd.read_csv(r"C:\Users\Dell\Documents\My ML Projects\Hybrid Recommendation System\data\processed_ratings.csv")

print("Movies shape:",movies_df.shape)
print("Ratings shape:",df.shape)

Movies shape: (10772, 3)
Ratings shape: (2128006, 5)


**Content Based Model**

In [3]:
#clean genres column
movies_df['genres'] = movies_df['genres'].str.replace('|',' ')


In [4]:
#Buid TF-IDF matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies_df['genres'])


In [5]:
#Compute cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix,tfidf_matrix)


In [6]:
#Build index mapping for titles
indices = pd.Series(movies_df.index,index=movies_df['title']).drop_duplicates()

In [7]:
#Define Content-Based Recommendation Function

def recommend_content(title,movies_df,cosine_sim,indices,n=10):
    if title not in indices:
        return [] #handle cold start
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1] #exculde the movie itself
    movie_indices = [i[0] for i in sim_scores]
    return movies_df['title'].iloc[movie_indices].tolist()

**Collaborative Filtering Model(item-based)**

In [8]:
#Reduce dataset to top 2000 movies for speed
top_movies = df['title'].value_counts().head(2000).index
df_filtered = df[df['title'].isin(top_movies)]

In [9]:
#Create User-Item Matrix 

user_item_matrix = df_filtered.pivot_table(
    index='userId',
    columns='title',
    values='rating'
).fillna(0)

In [10]:
#compute item-item similarity
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

In [11]:
#Define Collaborative Recommendation Function
def recommend_cf(movie_name,n=10):
    if movie_name not in item_similarity_df.columns:
        return []#handle cold start
    similar_scores = item_similarity_df[movie_name]
    similar_scores = similar_scores.sort_values(ascending=False)
    return similar_scores.index[1:n+1].tolist()#exclude the movie itself
    

In [12]:
def hybrid_recommend(title,movies_df,cosine_sim,indices,n=10,weight_content=0.4,weight_cf=0.6):
    
    #Get content-based recommendations
    content_rec = recommend_content(title,movies_df,cosine_sim,indices, n=20)

    #Get collaborative recommendations
    cf_rec = recommend_cf(title, n=20)

    #Weighted scoring(basic approach:give priority to CF if available)
    combined = []
    seen = set()

    #Add CF first with weight
    for movie in cf_rec:
        if movie not in seen:
            combined.append(movie)
            seen.add(movie)
        
    #Then add content recommendations
    for movie in content_rec:
        if movie not in seen:
            combined.append(movie)

    return combined[:n]

Test Hybrid Model

In [13]:
hybrid_recommend("Toy Story (1995)",movies_df,cosine_sim,indices,n=10)

['Jurassic Park (1993)',
 'Back to the Future (1985)',
 'Forrest Gump (1994)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Toy Story 2 (1999)',
 'Men in Black (a.k.a. MIB) (1997)',
 'Matrix, The (1999)',
 'Star Wars: Episode VI - Return of the Jedi (1983)',
 'Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)',
 'Star Wars: Episode V - The Empire Strikes Back (1980)']

Hybrid_model with weights


In [14]:
def hybrid_weighted(title, movies_df, cosine_sim, indices, item_similarity_df, n=10, weight_content=0.4, weight_cf=0.6):
    
    # ---------------- CONTENT ----------------
    content_scores = {}   # ✅ ALWAYS initialize

    if title in indices:
        idx = indices[title]
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        for i, score in sim_scores[1:200]:
            movie = movies_df['title'].iloc[i]
            content_scores[movie] = score

    # ---------------- CF ----------------
    cf_scores = {}   # ✅ ALWAYS initialize

    if title in item_similarity_df.columns:
        similar_series = item_similarity_df[title].sort_values(ascending=False)
        similar_series = similar_series.drop(title, errors='ignore')
        
        for movie, score in similar_series.head(200).items():
            cf_scores[movie] = score

    # ---------------- COMBINE ----------------
    combined_scores = {}

    all_movies = set(list(content_scores.keys()) + list(cf_scores.keys()))

    for movie in all_movies:
        c_score = content_scores.get(movie, 0)
        cf_score = cf_scores.get(movie, 0)

        combined_scores[movie] = (weight_content * c_score) + (weight_cf * cf_score)

    # ---------------- SORT ----------------
    sorted_movies = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
    print("Content count:", len(content_scores))
    print("CF count:", len(cf_scores))
    print("Total combined movies:", len(all_movies))

    return [movie for movie, score in sorted_movies[:n]]

Test weighted Hybrid

In [15]:
test_movie = "Toy Story (1995)"
top_n = 10

recommendations = hybrid_weighted(
    title=test_movie,
    movies_df=movies_df,
    cosine_sim=cosine_sim,
    indices=indices,
    item_similarity_df=item_similarity_df,
    n=top_n
)

print("Recommendations for", test_movie)
for i, movie in enumerate(recommendations, 1):
    print(f"{i}. {movie}")



Content count: 198
CF count: 200
Total combined movies: 386
Recommendations for Toy Story (1995)
1. Toy Story 2 (1999)
2. Monsters, Inc. (2001)
3. Shrek (2001)
4. Bug's Life, A (1998)
5. Finding Nemo (2003)
6. Who Framed Roger Rabbit? (1988)
7. Aladdin (1992)
8. Incredibles, The (2004)
9. Jumanji (1995)
10. Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)


Experiment withweights


In [16]:
hybrid_weighted(test_movie,movies_df,cosine_sim,indices,item_similarity_df,n=10,weight_content=0.6,weight_cf=0.4)

Content count: 198
CF count: 200
Total combined movies: 386


['Toy Story 2 (1999)',
 'Monsters, Inc. (2001)',
 'Shrek (2001)',
 "Bug's Life, A (1998)",
 'Finding Nemo (2003)',
 'Who Framed Roger Rabbit? (1988)',
 'Incredibles, The (2004)',
 'Aladdin (1992)',
 'Jumanji (1995)',
 "Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)"]

In [17]:
hybrid_weighted(test_movie,movies_df,cosine_sim,indices,item_similarity_df,n=10,weight_content=0.3,weight_cf=0.7)

Content count: 198
CF count: 200
Total combined movies: 386


['Toy Story 2 (1999)',
 'Shrek (2001)',
 'Monsters, Inc. (2001)',
 "Bug's Life, A (1998)",
 'Aladdin (1992)',
 'Finding Nemo (2003)',
 'Who Framed Roger Rabbit? (1988)',
 'Incredibles, The (2004)',
 'Jumanji (1995)',
 "Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)"]

In [2]:
from src.hybrid import hybrid_weighted
from src.content_based import build_tfidf_matrix, build_cosine_sim, build_indices
from src.collaborative import build_user_item_matrix, build_item_similarity
from src.data_preprocessing import load_and_preprocess_movies, load_ratings

# Load data
movies_df = load_and_preprocess_movies()
ratings_df = load_ratings()

# ---------------- REDUCE DATA ----------------
merged_df = ratings_df.merge(movies_df, on='movieId')

top_movies = merged_df['title'].value_counts().head(1000).index

movies_df = movies_df[movies_df['title'].isin(top_movies)]
ratings_df = ratings_df[ratings_df['movieId'].isin(movies_df['movieId'])]

top_users = ratings_df['userId'].value_counts().head(3000).index
ratings_df = ratings_df[ratings_df['userId'].isin(top_users)]

# ---------------- CONTENT ----------------
tfidf_matrix = build_tfidf_matrix(movies_df)
cosine_sim = build_cosine_sim(tfidf_matrix)
indices = build_indices(movies_df)

# ---------------- CF ----------------
user_item_matrix = build_user_item_matrix(ratings_df)
item_similarity_df = build_item_similarity(user_item_matrix)

# ---------------- HYBRID ----------------
print(hybrid_weighted("Toy Story (1995)", movies_df, cosine_sim, indices, item_similarity_df, n=10))

['Toy Story 2 (1999)', 'Monsters, Inc. (2001)', 'Shrek (2001)', "Bug's Life, A (1998)", 'Who Framed Roger Rabbit? (1988)', 'Antz (1998)', 'Finding Nemo (2003)', 'Incredibles, The (2004)', 'Chicken Run (2000)', 'Jumanji (1995)']
